# Concept segmentation in images with SAM 3


Say what you are looking for, and SAM 3 returns **every instance of it** in the image —
mask, box and score each. No clicking required, and no fixed label set: the concept is a
phrase you write.

That is the difference from SAM 2, which segments *the thing you pointed at* and has no idea
what it is. Here you never point unless you want to.

| you want | prompt | what comes back |
|---|---|---|
| every instance of a described thing | `ConceptPrompt(text="wheel")` | one mask per instance, keyed by score |
| the same, but biased toward one example | `geometry=GeometryPrompt.exemplar_box(xyxy)` | same instances, the boxed one scored up |
| the same, biased *away* from one | `GeometryPrompt.exemplar_box(xyxy, label=0)` | the boxed instance scored down |
| the same, from a point | `GeometryPrompt.exemplar_point((x, y))` | same instances, re-scored around the point |
| to search with no phrase at all | `predictor.PLACEHOLDER` + a concept prompt | detection under this lineage's box-only caption |
| "is this concept here at all?" | any of the above | `result.presence`, 0..1, before you look at instances |
| to prompt one image repeatedly | `predictor.encode(image)`, then pass that to `process` | the re-encode is skipped |

Everything here is **concept-driven**: you describe, the model finds every match. Marking a
spot only says "more like *this*". To get back one specific object instead — the SAM 2
gesture — pass `GeometryPrompt.click` / `.box` / `.mask` as `geometry=` and drop the concept,
which is section 7. Same predictor, same verb; the prompts you build decide which half of the
model answers.

This notebook covers text prompting, what `confidence_threshold` actually does, presence,
box and click exemplars, prompting with geometry and no phrase at all, selecting a single
object with geometry alone, and the SAM 3.1 multiplex predictor.

For the same model over a video, tracking each instance across frames, see
[`sam3_video_predictor_example.ipynb`](./sam3_video_predictor_example.ipynb).


## Set-up

SAM 3 weights are access-gated — request them via [Meta AI](https://ai.meta.com/sam), then
`pixi run download-sam3` and `pixi run download-sam3-1`. SAM 3 does its preprocessing on the
GPU, so a CUDA device is required rather than merely recommended.


In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

if os.path.isdir("notebooks"):     # started from the repo root rather than notebooks/
    sys.path.insert(0, "notebooks")
import nb_utils as nb  # shared plotting helpers — see notebooks/nb_utils.py

nb.use_repo_root()     # so "configs/..." and "checkpoints/..." resolve
nb.use_dark(False)     # light figures; see nb_utils.use_dark for PyCharm's inversion
device = nb.pick_device(require_cuda=True)

nb.require_checkpoints(("checkpoints/sam3.pt", "pixi run download-sam3"))


In [ ]:
from sam.build_sam import build_sam3, build_sam3_multiplex
from sam.prompts import ConceptPrompt, GeometryPrompt

predictor = build_sam3("configs/sam3/sam3.yaml", "./checkpoints/sam3.pt", device=device)
print("ready")


Two images, and one helper that draws whatever `predict` returns.


In [ ]:
TRUCK = np.array(Image.open("notebooks/images/truck.jpg").convert("RGB"))
GROCERIES = np.array(Image.open("notebooks/images/groceries.jpg").convert("RGB"))
print("truck:", TRUCK.shape, " groceries:", GROCERIES.shape)


def show(image, result, title, extra=None, width=9.0):
    """Draw every detected instance: mask, box, and the score above it."""
    masks = (result.masks_logits > 0).cpu().numpy()
    boxes = result.boxes.float().cpu().numpy()
    scores = result.scores.float().cpu().numpy()

    fig, ax = plt.subplots(figsize=(width, width * image.shape[0] / image.shape[1]))
    ax.imshow(image)
    ax.axis("off")
    caption = f"{title}\n{len(scores)} instance(s)"
    if result.presence is not None:   # None on the object path: no concept, nothing to be present
        caption += f" · presence {result.presence:.3f}"
    ax.set_title(caption)
    for i, (mask, box, score) in enumerate(zip(masks, boxes, scores)):
        nb.show_mask(mask, ax, obj_id=i)
        nb.show_box(box, ax, obj_id=i, label=f"{score:.2f}")
    if extra is not None:
        extra(ax)
    plt.tight_layout()
    plt.show()
    plt.close(fig)


## 1. Text — every instance of a concept

One call. The phrase is the whole prompt.


In [ ]:
result = predictor.process(GROCERIES, concept=ConceptPrompt(text="paper bag"))

print("masks_logits:", tuple(result.masks_logits.shape), "  (N, H, W) — > 0 is foreground")
print("boxes:       ", tuple(result.boxes.shape), "     — xyxy pixels")
print("scores:      ", result.scores.float().cpu().numpy().round(3))
print("presence:    ", round(result.presence, 3))
print("instance_ids:", result.instance_ids.cpu().numpy())

show(GROCERIES, result, 'ConceptPrompt(text="paper bag")')


Every field is per-instance except `presence`, which is one number for the image.

`masks_logits` are *logits*: positive means foreground, so `result.masks_logits > 0` is your
binary mask. The boxes are the detector's own, in pixels — you do not derive them from the
masks the way the video notebooks have to.

The concept is a phrase, not a class, so the wording is yours to choose. `"grocery bag"`
finds the same four bags here; a phrase the image does not contain finds nothing at all,
which is section 3.


In [ ]:
result = predictor.process(TRUCK, concept=ConceptPrompt(text="wheel"))
show(TRUCK, result, 'ConceptPrompt(text="wheel")')


Four, not two: the two far-side wheels are visible under the body, and each is its own
instance with its own score. Nothing told the model how many to expect.


## 2. `confidence_threshold` — how sure is sure enough

`process` keeps instances whose presence-weighted score clears the threshold. The default,
0.5, is not a magic number; it is a dial, and the useful range depends on the image.

Sweeping it means prompting one image five times, and encoding the image is about 70% of a
`process` call while depending on none of the prompt. `encode` pays that once: hand the
result back to `process` in place of the array and only the detect runs. Measured on this
sweep, 1862 ms of five `process` calls becomes 490 ms — with results that are bit-identical,
which `tests/test_image_process.py` pins.


In [ ]:
bags = predictor.encode(GROCERIES)   # one encode, five prompts

for t in (0.1, 0.3, 0.5, 0.7, 0.9):
    r = predictor.process(bags, concept="paper bag", confidence_threshold=t)
    print(f"threshold {t}: {int(r.boxes.shape[0]):2} instance(s)  "
          f"scores={r.scores.float().cpu().numpy().round(2)}")


The four real bags sit around 0.89–0.90, so anything from 0.3 to 0.7 returns exactly
them. Drop to 0.1 and you also collect a tail of ~0.1-scored guesses; raise to 0.9 and you
keep only the single best. A wide plateau like this one means the model is confident — a
threshold that changes the count continuously would mean the opposite.


In [ ]:
loose = predictor.process(bags, concept="paper bag", confidence_threshold=0.1)
show(GROCERIES, loose, "confidence_threshold=0.1 — the tail is visible")


## 3. Presence — is the concept in this image at all

`presence` answers that before you look at a single instance, which is what you want when
the answer is often "no".


In [ ]:
for name, image in (("groceries", GROCERIES), ("truck", TRUCK)):
    r = predictor.process(image, concept=ConceptPrompt(text="elephant"))
    print(f"{name:10} 'elephant': {int(r.boxes.shape[0])} instance(s), "
          f"presence {r.presence:.3f}")


Zero instances and a presence of essentially zero — the model is not hedging. Compare
with 0.99 for `"wheel"` on the truck. Use `presence` as the gate and `scores` to rank what
survives it.


## 4. Box exemplar — point at one, keep the concept

Pass a `GeometryPrompt` as `geometry` and the box becomes an *example* of what you mean, not
a selection. The concept still drives detection; the box biases it.


In [ ]:
REAR_WHEEL = (450.0, 620.0, 670.0, 840.0)  # xyxy pixels, the rear wheel

plain = predictor.process(TRUCK, concept=ConceptPrompt(text="wheel"))
hinted = predictor.process(
    TRUCK, concept=ConceptPrompt(text="wheel"),
    geometry=GeometryPrompt.exemplar_box(REAR_WHEEL),
)

print("no exemplar:", plain.scores.float().cpu().numpy().round(3),
      f"presence {plain.presence:.3f}")
print("box exemplar:", hinted.scores.float().cpu().numpy().round(3),
      f"presence {hinted.presence:.3f}")

show(TRUCK, hinted, "wheel + positive box exemplar (dashed)",
     extra=lambda ax: nb.show_box(REAR_WHEEL, ax, label="exemplar", style="--"))


Still four wheels — the exemplar did not narrow the search to one object. What moved is
confidence: the boxed wheel scores higher than it did unprompted, and presence goes to 1.00.
That is the contract: an exemplar is an example, never a selection. If you want *only* that
object, drop the concept and pass the geometry on its own — section 7.

`label=0` inverts the bias — "everything matching the concept **except** this one".


In [ ]:
against = predictor.process(
    TRUCK, concept=ConceptPrompt(text="wheel"),
    geometry=GeometryPrompt.exemplar_box(REAR_WHEEL, label=0),
)
print("negative exemplar:", against.scores.float().cpu().numpy().round(3))

show(TRUCK, against, "wheel + negative box exemplar (dashed)",
     extra=lambda ax: nb.show_box(REAR_WHEEL, ax, label="not this", style="--"))


The boxed wheel drops to 0.66 while the others hold above 0.87 — but it is still on
screen, because 0.66 clears the default threshold of 0.5. A negative exemplar **reweights; it
does not delete**. Combine it with the threshold and the instance goes away, while the same
threshold leaves the unprompted prediction untouched:


In [ ]:
for t in (0.5, 0.7):
    plain_t = predictor.process(TRUCK, concept=ConceptPrompt(text="wheel"), confidence_threshold=t)
    against_t = predictor.process(
        TRUCK, concept="wheel", confidence_threshold=t,
        geometry=GeometryPrompt.exemplar_box(REAR_WHEEL, label=0),
    )
    print(f"threshold {t}: no exemplar -> {int(plain_t.boxes.shape[0])} wheels, "
          f"negative exemplar -> {int(against_t.boxes.shape[0])} wheels")

show(TRUCK, against_t, "negative exemplar at confidence_threshold=0.7 — the boxed wheel is gone",
     extra=lambda ax: nb.show_box(REAR_WHEEL, ax, label="not this", style="--"))


At 0.7 the plain prompt still returns all four (its weakest is 0.89); the negated one
returns three. The boxed wheel is the only instance the label removed, which is what makes it
a bias on that instance rather than a global confidence shift.

That the label reaches the model at all is not taken on trust: it indexes
`geometry_encoder.label_embed`, an `nn.Embedding(2, d)` summed into the box token, and both
signs are pinned against captured upstream goldens by
[`tests/parity/test_sam3_box_prompt_parity.py`](../tests/parity/test_sam3_box_prompt_parity.py)
— the negative stem fails outright if `boxes_labels` is ignored.


## 5. Point exemplar

Same idea, one point instead of four numbers. `exemplar_point` is the point form of
`exemplar_box` — no `obj_id`, because it selects nothing.


In [ ]:
CLICK = (560.0, 730.0)  # centre of the rear wheel

clicked = predictor.process(
    TRUCK, concept=ConceptPrompt(text="wheel"),
    geometry=GeometryPrompt.exemplar_point(CLICK),
)
print("point exemplar:", clicked.scores.float().cpu().numpy().round(3),
      f"presence {clicked.presence:.3f}")

show(TRUCK, clicked, "wheel + point exemplar",
     extra=lambda ax: nb.show_points([CLICK], [1], ax))


A point moves the scores too, but not the way the box did: here every wheel comes back
a little *lower*, the marked one included (0.95 unprompted against 0.89 with the point). A box
says where **and** how big; a lone point says less, and this detector was already sure.

`label=0` is weaker still. Where a negative *box* pushed its wheel from 0.95 down to 0.66,
a negative point leaves all four sitting at 0.89–0.96 — it does not have the extent to say
which instance you mean. Reach for the box when you have one.


## 6. No phrase at all — the box-only caption

Sometimes you have an example and no words for it. SAM 3 handles that with a **placeholder
caption**: upstream encodes a fixed string whenever geometry arrives with no text, and the
string is lineage-specific — `"visual"` on base SAM 3, `"<text placeholder>"` on the 3.1
multiplex. Ask for it by name rather than typing it, because the wrong lineage's caption
returns nothing at all:


In [ ]:
print("this lineage's box-only caption:", repr(predictor.BOX_ONLY_CAPTION))

boxed_only = predictor.process(
    TRUCK, concept=predictor.PLACEHOLDER,
    geometry=GeometryPrompt.exemplar_box(REAR_WHEEL),
)
print("PLACEHOLDER + box:", boxed_only.scores.float().cpu().numpy().round(3),
      f"presence {boxed_only.presence:.3f}")

show(TRUCK, boxed_only, "no phrase, one box — find things like this one",
     extra=lambda ax: nb.show_box(REAR_WHEEL, ax, label="exemplar", style="--"))


Four wheels, the boxed one strongest at 0.96 — the box said what to look for and the
caption carried no opinion of its own. This is the same route the video predictor opens with
`start_concept_session(PLACEHOLDER)`.

Two things worth knowing about it. A point works but says less — three instances here rather
than four, at lower scores, for the same reason a negative point was weak in section 5. And
the caption *alone*, with no geometry, is not a prompt at all: it detects generic objects at
around 0.5, which is noise rather than an answer.


In [ ]:
for label, geometry in (
    ("PLACEHOLDER + point", GeometryPrompt.exemplar_point(CLICK)),
    ("PLACEHOLDER alone  ", None),
):
    r = predictor.process(TRUCK, concept=predictor.PLACEHOLDER, geometry=geometry)
    print(f"{label}: {int(r.boxes.shape[0]):2} instance(s), presence {r.presence:.3f}, "
          f"top score {float(r.scores.max()) if len(r.scores) else 0:.2f}")


## 6b. The two paths do not mix

`concept=` and tracker geometry each answer a different question, and one result cannot
honestly hold both: detection mints ids `0..N-1` while `click` / `box` / `mask` carry the
`obj_id` you chose, so merging them would alias two different things into one
`instance_ids` array. Asking for both raises instead:


In [ ]:
for geometry in (GeometryPrompt.box(1, REAR_WHEEL), GeometryPrompt.click(1, CLICK)):
    try:
        predictor.process(TRUCK, concept="wheel", geometry=geometry)
    except ValueError as exc:
        print(f"{type(exc).__name__}: {exc}")
        break


Drop the concept and the same prompts select objects (section 7); keep the concept and
use `exemplar_*` geometry and they bias the search (sections 4–6). Two more limits worth
knowing:

- **Mask prompts have no concept form.** `GeometryPrompt.mask(obj_id, mask)` prompts the
  tracker; there is no `exemplar_mask`, because neither SAM 3 checkpoint ships
  `mask_encoder` weights. Building one on the detector route raises at construction.
- **Several exemplars go in one prompt.** `exemplar_boxes([...], labels=[...])` and
  `exemplar_points([...], labels=[...])` build those; passing two separate exemplar prompts
  raises, because they describe one search rather than one object each.


In [ ]:
pair = predictor.process(TRUCK, concept="wheel", geometry=GeometryPrompt.exemplar_boxes(
    [REAR_WHEEL, (1396.0, 560.0, 1627.0, 776.0)],   # both road wheels as examples
))
print("two exemplars:", pair.scores.float().cpu().numpy().round(3),
      f"presence {pair.presence:.3f}")

show(TRUCK, pair, "wheel + two box exemplars")


## 7. Selecting one object — geometry without a concept

Everything above detects. When you want SAM 2's contract instead — *this* object, the one I
marked, and nothing else — drop the concept and pass tracker geometry:

```python
predictor.process(image, geometry=GeometryPrompt.click(1, (x, y)))
```

Same predictor, same weights, same call. `build_sam3` loads the tracker (12M parameters
against the detector's 3.5 GB), so the object half of SAM 3 is right here rather than behind
a second model. Which path runs is decided by which arguments you passed, never guessed:
`concept=` detects, `click` / `box` / `mask` select.


In [ ]:
picked = predictor.process(TRUCK, geometry=GeometryPrompt.box(1, REAR_WHEEL))

print("instance_ids:", picked.instance_ids.tolist(), " (the id you passed)")
print("presence:    ", picked.presence, "     (no concept, so nothing to be present)")
print("foreground px:", int((picked.masks_logits > 0).sum()))

show(TRUCK, picked, "geometry only — one object, the one you asked for",
     extra=lambda ax: nb.show_box(REAR_WHEEL, ax, label="prompt", style="--"))


One mask, one id, no opinion about the other three wheels — which is exactly what the
concept path cannot give you. Prompts compose: pass a list to select several objects at once,
each under its own id.


In [ ]:
pair = predictor.process(TRUCK, geometry=[
    GeometryPrompt.click(1, CLICK),
    GeometryPrompt.box(2, (1396.0, 560.0, 1627.0, 776.0)),
])
print("ids:", pair.instance_ids.tolist(),
      "| px each:", [int(x) for x in (pair.masks_logits > 0).sum(dim=(1, 2))])

show(TRUCK, pair, "two prompts, two objects, your ids")


The result type is the same `Sam3DetectionResult` either way, which is what lets one
verb serve both paths:

| | `process(image, concept=...)` | `process(image, geometry=click/box/mask)` |
|---|---|---|
| answers | "find every X" | "mask the thing I marked" |
| ids | minted by the detector, 0..N-1 | the `obj_id` you passed |
| `presence` | 0..1 for the phrase | `None` |
| boxes | predicted by the detector | derived from the returned masks |
| geometry means | an example to bias the search | the object itself |

Mixing the two in one call raises rather than merging: detector ids start at 0 and yours are
whatever you chose, so one `instance_ids` array cannot honestly hold both. Make two calls.

This is a single image, so each prompt is a fresh start. Give the same geometry to
[`sam3_video_predictor_example.ipynb`](./sam3_video_predictor_example.ipynb)'s
`start_session()` and the object is tracked across frames instead — the mask on frame 0 is
the same one, to within the difference the video frame loader's resize makes (IoU 0.997).


## 8. SAM 3.1 (multiplex) — the same call

SAM 3.1 is a different checkpoint and a different config; the calling code is unchanged. Its
scoring differs internally — the score is the joint logit with presence already folded in,
and the boxes are derived from the masks rather than predicted directly — so expect the same
instances with slightly different numbers.

The checkpoints are several GB each and do not comfortably co-reside, so free the resident one first.


In [ ]:
nb.free(globals(), "predictor")

nb.require_checkpoints(("checkpoints/sam3.1_multiplex.pt", "pixi run download-sam3-1"))
mux = build_sam3_multiplex(
    "configs/sam3/sam3.1.yaml", "./checkpoints/sam3.1_multiplex.pt", device=device,
)

mux_result = mux.process(TRUCK, concept="wheel")
print("SAM 3.1 scores:", mux_result.scores.float().cpu().numpy().round(3),
      f"presence {mux_result.presence:.3f}")

show(TRUCK, mux_result, "SAM 3.1 multiplex — same prompt, same API")


Same four wheels, boxes within a pixel or two of the base model's. Both paths work on
this lineage too — `process(TRUCK, geometry=GeometryPrompt.box(1, REAR_WHEEL))` decodes
through SAM 3.1's own interactive head — so the choice of checkpoint changes the weights, not
the API. The one difference: a mask prompt has no multiplex path and raises.


In [ ]:
mux_picked = mux.process(TRUCK, geometry=GeometryPrompt.box(1, REAR_WHEEL))
print("ids:", mux_picked.instance_ids.tolist(),
      "| px:", int((mux_picked.masks_logits > 0).sum()),
      "| presence:", mux_picked.presence)

show(TRUCK, mux_picked, "SAM 3.1 — the object you marked",
     extra=lambda ax: nb.show_box(REAR_WHEEL, ax, label="prompt", style="--"))


## What you can do from here

| | how |
|---|---|
| Find every instance of a thing | `predictor.process(image, concept=ConceptPrompt(text="..."))` |
| Tighten or loosen the result set | `confidence_threshold=` (0.5 default) |
| Prompt one image several times | `enc = predictor.encode(image)` then `process(enc, ...)` |
| Ask whether it is there at all | `result.presence` |
| Bias toward an example | `geometry=GeometryPrompt.exemplar_box((x0, y0, x1, y1))` |
| Bias away from one | the same, `label=0` |
| Bias from a point | `geometry=GeometryPrompt.exemplar_point((x, y))` |
| Search with no phrase | `predictor.PLACEHOLDER` + `exemplar_box(...)` |
| Binary masks | `result.masks_logits > 0` |
| Prompt with a box and no phrase | `ConceptPrompt(text="")` + `exemplar_box(...)` |
| Select one object, not a concept | `process(image, geometry=GeometryPrompt.click(1, (x, y)))` |
| Run SAM 3.1 instead | `build_sam3_multiplex` + `sam3.1.yaml` + `sam3.1_multiplex.pt` |
| Track those instances across a video | [`sam3_video_predictor_example.ipynb`](./sam3_video_predictor_example.ipynb) |

The rule of thumb: `exemplar_*` prompts describe what to find, plain ones pick out one object.
Both live on this predictor, and `process` runs whichever one you built.
